# ATLOP vs ATLOP+GREP — Colab 재현·비교 노트북

> **최종 업데이트**: 2026-07-13: ATLOP baseline과 GREP(Zhang, Yan & Cheng, "Document-Level Relation Extraction with Global Relations and Entity Pair Reasoning," ACL Findings 2025) 재현을 각각 학습해 같은 dev set·같은 스코어러로 F1/Ign F1을 비교하는 Colab 노트북 최초 작성.

이 노트북은 `Scripts/atlop/`(ATLOP baseline)과 `Scripts/grep/`(ATLOP 위에 GREP의 세 기법을 얹은 확장)을 **같은 데이터·같은 하이퍼파라미터·같은 스코어러**로 학습·평가해서, GREP 기법을 추가했을 때 **F1이 얼마나 달라지는지**만 비교합니다. GREP이 추가하는 세 기법:

1. **Evidence Extraction Module** — pair별 문장 attention을 KL-divergence loss로 gold evidence 문장에 맞춰 학습해, 불필요한 문장을 걸러내고 핵심 근거 문장에 집중
2. **Global Relation Prediction** — 문서 전체에서 존재 가능한 관계를 먼저 예측(BCE loss)해 pair별 로짓에 더해줌으로써 애초에 존재하지 않는 관계를 걸러냄
3. **Inference Fusion** — evidence loss로 학습한 모델(`model_full`)과 evidence loss 없이 학습한 모델(`model_no_evi`)을 준비해, 첫 모델이 예측한 핵심 문장만으로 만든 pseudo-document를 두 번째 모델에 다시 태워 두 추론 결과를 융합

(Entity Pair Reasoning Graph는 GREP 아키텍처 자체에 항상 포함되는 구조라 on/off 스위치가 없고, 위 세 기법과 함께 `model_full`에 이미 녹아 있습니다.)

**공정 비교 설계**: 두 모델 모두 `train_distant`(원거리 감독, 노이즈 있음) 없이 `train_annotated`만으로 학습합니다 — GREP 논문의 DocRED 설정과 동일하고, `results/comparison.md`에서도 "공정 비교용 annotated 단독 실행(`--distant_mode none`)"을 권장합니다. `results/comparison.md`에 기록된 ATLOP 공식 baseline(dev F1 61.71 / Ign F1 59.86)은 distant 사전학습을 포함한 수치라 이 노트북의 수치와는 조건이 다릅니다 — 이 노트북이 보려는 건 절대 성능이 아니라 **같은 조건에서 GREP 모듈이 만드는 F1 차이**입니다.

기본값은 **빠른 데모 스케일**(`--limit_docs`, epoch 축소)입니다. 논문 스케일 전체 재현(`train_annotated` 3,053 문서 × 30 epoch, GREP은 모델 2개라 대략 2배 시간)으로 바꾸는 방법은 "1. 공통 하이퍼파라미터" 셀에 있습니다.

## 0. 사전 준비 (로컬에서 1회)

이 저장소의 `Scripts/grep/`와 `Scripts/atlop/preprocess.py` 확장은 아직 GitHub에 push되지 않은 로컬 변경일 수 있습니다. Colab에서 `git clone`으로 받는 대신, 실행에 필요한 최소 파일만 압축한 `colab_bundle.zip`(프로젝트 루트, `.gitignore`에 이미 등록된 이름)을 사용합니다. 포함 내용: `Scripts/atlop`, `Scripts/grep`, `Scripts/eval`, `data/`, `docred_data/data/{train_annotated,dev,rel_info}.json`, `requirements.txt` — 이번 비교에서 쓰지 않는 `train_distant.json`(437MB)은 제외해서 3.6MB로 가볍습니다.

1. `colab_bundle.zip`을 Google Drive에 업로드하세요 (예: `내 드라이브/Information_Extraction/colab_bundle.zip`).
2. 아래 셀의 `BUNDLE_PATH`를 실제 업로드 경로로 맞추세요.
3. 로컬 코드를 수정했다면(예: `Scripts/grep/re_model.py` 튜닝) `colab_bundle.zip`을 다시 만들어 재업로드해야 반영됩니다.
4. Colab 런타임을 **GPU**로 설정하세요 (런타임 > 런타임 유형 변경 > T4 GPU).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BUNDLE_PATH = "/content/drive/MyDrive/Information_Extraction/colab_bundle.zip"  # 실제 업로드 경로로 수정
PROJECT_ROOT = "/content/Information_Extraction"

!rm -rf {PROJECT_ROOT}
!mkdir -p {PROJECT_ROOT}
!unzip -q "{BUNDLE_PATH}" -d {PROJECT_ROOT}
!echo '--- top level ---' && ls {PROJECT_ROOT}
!echo '--- data ---' && ls {PROJECT_ROOT}/docred_data/data

In [ ]:
# Colab 기본 torch(CUDA 빌드에 맞춰 이미 설치됨)는 건드리지 않고, 버전이 고정된 나머지 패키지만 맞춘다.
# requirements.txt 전체를 pip install -r 하면 torch가 CPU 전용 등 다른 빌드로 교체되어
# Colab GPU와 안 맞을 수 있으므로 일부러 피한다.
!pip install -q "transformers==4.57.6" "accelerate==1.10.1"

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# 배선 검증 (다운로드 없는 랜덤 소형 BERT, 정확도 아님) -- 실제 학습 전에 문제가 있으면 여기서 먼저 걸린다.
!python -m Scripts.atlop.smoke_test
!python -m Scripts.grep.smoke_test

## 1. 공통 하이퍼파라미터

`LIMIT_DOCS`/`EPOCHS`를 줄인 빠른 비교용 기본값입니다. `--limit_docs`는 `train`/`dev` 둘 다 같은 값으로 제한합니다(ATLOP·GREP 스크립트 공통 동작). 논문 스케일로 돌리려면 `LIMIT_DOCS = 0`(전체 train_annotated 3,053 / dev 998), `EPOCHS = 30`으로 바꾸고 아래 셀부터 다시 실행하세요. GREP은 `model_full`/`model_no_evi` 두 모델을 각각 학습하므로 ATLOP보다 대략 2배 걸립니다.

In [ ]:
MODEL_NAME = "bert-base-cased"
LIMIT_DOCS = 300   # 0 = 전체(train_annotated 3,053 / dev 998). 빠른 비교용 기본값.
EPOCHS = 4         # 논문 기본값 30. 빠른 비교용으로 축소.
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
SEED = 66
RUN_NAME_ATLOP = "atlop_demo"
RUN_NAME_GREP = "grep_demo"

# ATLOP/GREP 두 argparser에 공통으로 있는 플래그만 모아서 재사용 -- 두 모델이
# 정확히 같은 데이터/스케일/시드로 학습되도록 보장하는 것이 공정 비교의 핵심이다.
COMMON_ARGV = [
    "--model_name_or_path", MODEL_NAME,
    "--train_split", "train_annotated",
    "--dev_split", "dev",
    "--epochs", str(EPOCHS),
    "--train_batch_size", str(TRAIN_BATCH_SIZE),
    "--eval_batch_size", str(EVAL_BATCH_SIZE),
    "--seed", str(SEED),
    "--limit_docs", str(LIMIT_DOCS),
]

## 2. ATLOP baseline 학습

GREP과 조건을 맞추기 위해 `--distant_mode none`(annotated 단독, distant supervision 미사용)으로 돌립니다. `train_re.train()`은 마지막 epoch의 dev 지표 dict(`f1`/`ign_f1`/`precision`/`recall` 등)를 반환합니다.

In [ ]:
from Scripts.atlop import train_re

atlop_args = train_re.build_argparser().parse_args(
    COMMON_ARGV + ["--distant_mode", "none", "--run_name", RUN_NAME_ATLOP]
)
atlop_metrics = train_re.train(atlop_args)
print(atlop_metrics)

## 3. GREP 학습 (`model_full` + `model_no_evi` + Inference Fusion)

`train_grep.train()`은 `{"model_full": ..., "fused": ...}`를 반환하도록 이 작업에서 `Scripts/grep/train_grep.py`의 마지막 `return`을 확장했습니다(원래는 fusion 결과만 반환 — CLI로 실행할 때는 반환값을 쓰지 않으므로 기존 동작에는 영향 없음). `model_full`은 Evidence Extraction Module + Global Relation Prediction까지 적용된 결과이고(Entity Pair Reasoning Graph는 항상 포함), `fused`는 여기에 Inference Fusion(Eq 22)까지 더한 최종 결과입니다.

빠른 데모에서는 `--sweep_gamma`(gamma 그리드 탐색, dev를 7번 더 평가)를 끄고 `gamma=0.0` 기본값을 씁니다. 논문 스케일로 돌릴 때는 `--sweep_gamma`를 켜는 것을 권장합니다(아래 3번째 코드 셀 주석 참고).

In [ ]:
from Scripts.grep import train_grep

grep_args = train_grep.build_argparser().parse_args(
    COMMON_ARGV + ["--run_name", RUN_NAME_GREP]
    # 논문 스케일에서는 위 리스트에 "--sweep_gamma"를 추가해 gamma를 dev에서 탐색할 것을 권장.
)
if grep_args.node_dim == 0:
    grep_args.node_dim = None  # train_grep.py __main__ 진입점과 동일한 처리

grep_metrics = train_grep.train(grep_args)
print(grep_metrics)

## 4. F1 비교

세 지점을 나란히 비교합니다: **ATLOP baseline** → **+ GREP 모듈(`model_full`)** → **+ Inference Fusion(`fused`)**. 같은 dev set·같은 스코어러(`Scripts/eval/scorer.py`)로 계산된 수치라 직접 비교할 수 있습니다.

In [ ]:
import pandas as pd

rows = [
    {"model": "ATLOP baseline", **atlop_metrics},
    {"model": "+ GREP modules (model_full)", **grep_metrics["model_full"]},
    {"model": "+ Inference Fusion (fused)", **grep_metrics["fused"]},
]
df = pd.DataFrame(rows)[["model", "f1", "ign_f1", "precision", "recall"]]
df[["f1", "ign_f1", "precision", "recall"]] *= 100
df["f1_delta_vs_atlop"] = df["f1"] - df.loc[0, "f1"]
df.round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

INK = "#0b0b0b"
MUTED = "#898781"
GRID = "#e1e0d9"
SERIES_F1 = "#2a78d6"    # categorical slot 1 (blue)
SERIES_IGN = "#1baf7a"   # categorical slot 2 (aqua)

labels = df["model"].tolist()
f1_vals = df["f1"].tolist()
ign_vals = df["ign_f1"].tolist()

x = np.arange(len(labels))
width = 0.32

fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars1 = ax.bar(x - width / 2, f1_vals, width, label="F1", color=SERIES_F1)
bars2 = ax.bar(x + width / 2, ign_vals, width, label="Ign F1", color=SERIES_IGN)

for bars in (bars1, bars2):
    for b in bars:
        ax.annotate(f"{b.get_height():.1f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                    textcoords="offset points", xytext=(0, 3), ha="center", fontsize=9, color=INK)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("dev score", color=MUTED)
ax.tick_params(colors=MUTED)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(MUTED)
ax.legend(frameon=False)
ax.set_title("ATLOP baseline vs. ATLOP + GREP modules (dev)", color=INK, fontsize=11)
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/results/atlop_vs_grep_comparison.png", dpi=150)
plt.show()

## 5. 결과물 및 논문 스케일 재현

- 예측: `results/{atlop_demo,grep_demo}_dev_predictions.json`
- 비교 그림: `results/atlop_vs_grep_comparison.png`
- 논문 스케일로 다시 돌리려면 "1. 공통 하이퍼파라미터" 셀에서 `LIMIT_DOCS = 0`, `EPOCHS = 30`으로 바꾸고 2~4번 섹션 셀을 다시 실행하세요. GREP 학습 셀에서는 `--sweep_gamma`를 추가하는 것을 권장합니다.
- 이 노트북은 CPU 스모크 테스트(`Scripts.atlop.smoke_test`/`Scripts.grep.smoke_test`)로 배선만 검증하고, 실제 성능은 GPU 학습 후에만 의미가 있습니다(두 README와 동일한 원칙).
- 최종 수치는 `results/comparison.md`(git 추적)에 옮겨 적어 팀과 공유하세요.